In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, datasets, transforms

In [3]:
import sys
print(sys.executable)

/home/iztihad/venvs/ml/bin/python


In [4]:
model_config = {
    "batch_size": 16,
    "input_size": 224,
    "architecture": "effiecientnet_b2",
    "learning_rate": 0.001,
    "epochs": 20,
    "pretrained":True
}

In [5]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])
    ]),

    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])
    ]),

    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])
    ])
}

train_dir = "../BanglaLekha_dataset_split/train"
val_dir = "../BanglaLekha_dataset_split/validation"
test_dir = "../BanglaLekha_dataset_split/test"


train_dataset = datasets.ImageFolder(root=train_dir, transform=data_transforms["train"])
val_dataset = datasets.ImageFolder(root=val_dir, transform=data_transforms["val"])
test_dataset = datasets.ImageFolder(root=test_dir, transform=data_transforms["test"])

train_dataloader = DataLoader(train_dataset, batch_size=model_config["batch_size"], shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=model_config["batch_size"], shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=model_config["batch_size"], shuffle=False)

In [7]:

efficientnet_b2 = models.efficientnet_b2(pretrained=True)


for param in efficientnet_b2.parameters():
    param.requires_grad = False

in_features = efficientnet_b2.classifier[1].in_features
efficientnet_b2.classifier[1] = nn.Linear(in_features, 84)

total_params = sum(p.numel() for p in efficientnet_b2.parameters())

gpu = torch.device("cuda")
efficientnet_b2 = efficientnet_b2.to(gpu)


In [8]:
print(total_params)

7819350


In [18]:
import fine_tuning as ft
ft.fine_tune(model=efficientnet_b2, model_name="efficientnet_b2", state="full") #Change the state for fine-tuning

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW([
    {"params": efficientnet_b2.classifier[1].parameters(), "lr": 1e-4, "weight-decay": 1e-4},
    {"params": efficientnet_b2.features.parameters(), "lr": 1e-5, "weight-decay": 1e-4},
    
])
epochs = model_config["epochs"]

In [11]:
def validate_model(model, val_dataloader):
    with torch.no_grad():
        model.eval()
        total = 0
        total_correct = 0

        for images, labels in val_dataloader:
            images = images.to(gpu)
            labels = labels.to(gpu)

            output = model(images)
            _, predicted = torch.max(output, 1)

            total = total + len(labels)
            total_correct = total_correct + (predicted == labels).sum().item()

        return total_correct/total 



In [12]:
def train_model(model, train_dataloader, val_dataloader, optimizer, criterion, epochs):
    
    max_val_accuracy = 0
    count = 0
    patience = 5

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, label in train_dataloader:
            images = images.to(gpu)
            label = label.to(gpu)

            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, label)
            loss.backward()
            optimizer.step()

            total_loss = total_loss + loss.item()
        
        val_accuracy = validate_model(efficientnet_b2, val_dataloader)

        if(val_accuracy > max_val_accuracy):
            max_val_accuracy = val_accuracy
            count = 0

            torch.save(model.state_dict(), "saved_parameters/efficientnet_b2.pth")

        else:
            count = count + 1

        if(count >= patience):
            break

        

        print(f"Epoch: {epoch + 1}, Training Loss: {total_loss/len(train_dataloader)}, Validation Accuracy: {val_accuracy}")

In [19]:
train_model(efficientnet_b2, train_dataloader, val_dataloader, optimizer, criterion, model_config["epochs"])

Epoch: 1, Training Loss: 0.23022915710756564, Validation Accuracy: 0.9332287834006876
Epoch: 2, Training Loss: 0.1946870016458441, Validation Accuracy: 0.9346763978527052
Epoch: 3, Training Loss: 0.17053029539964795, Validation Accuracy: 0.9360636950358888
Epoch: 4, Training Loss: 0.1492039915805905, Validation Accuracy: 0.9382351167139152
Epoch: 5, Training Loss: 0.1360131822032479, Validation Accuracy: 0.9373906749502382
Epoch: 6, Training Loss: 0.1221316728298801, Validation Accuracy: 0.9393811448217625
Epoch: 7, Training Loss: 0.10956512049522689, Validation Accuracy: 0.9386573375957536
Epoch: 8, Training Loss: 0.09782969689409786, Validation Accuracy: 0.9421557391881296
Epoch: 9, Training Loss: 0.0898478406459316, Validation Accuracy: 0.9381747994450811
Epoch: 10, Training Loss: 0.08027847077021152, Validation Accuracy: 0.9393811448217625
Epoch: 11, Training Loss: 0.07446457405882924, Validation Accuracy: 0.9384160685204174
Epoch: 12, Training Loss: 0.06843830840792296, Validation

In [20]:
efficientnet_b2.load_state_dict(torch.load("saved_parameters/efficientnet_b2.pth"))

<All keys matched successfully>

In [22]:
accuracy = validate_model(efficientnet_b2, test_dataloader)
print(f"Accuracy: {100 * accuracy}")

Accuracy: 93.8628375739734
